In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 19 — SHAP: A CONTABILIDADE DE CADA PREVISÃO

> "Não me diga que o time venceu. Diga-me como cada jogador contribuiu para o placar. Quero a contabilidade exata de cada ponto."
> — Um técnico obcecado por estatísticas

A importância global me deu um mapa geral, mas sem legendas para jornadas individuais. Para confiar plenamente no OncoClassify 2.0, eu precisava auditar *cada* previsão.

Lembrei da teoria dos jogos e do **Valor de Shapley**, do Nobel Lloyd Shapley: uma forma justa de distribuir os ganhos de um jogo cooperativo entre os jogadores. Em 2017, pesquisadores adaptaram a ideia para o *machine learning* — o **SHAP**. A previsão vira um "jogo": a previsão média é o ponto de partida, cada *feature* é um "jogador", e o SHAP calcula a contribuição exata de cada uma para mover a previsão da média até o resultado final. Era exatamente a contabilidade que eu procurava.

## SHAP: Teoria dos Jogos Para Explicar Modelos

O SHAP tem uma base matemática sólida, com propriedades desejáveis. A principal é a **eficiência local**: a soma dos valores SHAP de todas as *features*, mais o valor-base, é exatamente igual à previsão do modelo para aquele ponto. Ou seja, a explicação **fecha a conta** — nada some, nada aparece do nada.

Como o SVM é um modelo genérico (não baseado em árvores), usamos o `KernelExplainer`. Ele é computacionalmente caro, então explicamos uma amostra do teste, com um resumo do treino como referência (*background*).

#### Código 19.1: Calculando os Valores SHAP


In [ ]:
import numpy as np, pandas as pd, shap
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from oncoclassify_utils import X_train, y_train, FEATURES_MORFOLOGICAS
np.random.seed(42)

def info_mutua(X, y): return mutual_info_classif(X, y, random_state=42)
modelo = Pipeline([("selecao", SelectKBest(info_mutua, k=25)),
                   ("escala", StandardScaler()),
                   ("svm", SVC(kernel="rbf", C=10, gamma=0.01,
                               probability=True, random_state=42))]).fit(X_train, y_train)

# funcao que devolve P(Maligno) (classe 0) — o que queremos explicar
def prob_maligno(dados):
    return modelo.predict_proba(pd.DataFrame(dados, columns=FEATURES_MORFOLOGICAS))[:, 0]

background = shap.kmeans(X_train.values, 15)          # resumo do treino
explainer  = shap.KernelExplainer(prob_maligno, background)

amostra    = X_train.iloc[:40]                         # 40 pacientes do treino (cofre lacrado)
shap_values = explainer.shap_values(amostra.values, nsamples=100)
print(f"valor-base E[P(Maligno)] = {explainer.expected_value:.3f}")
print(f"formato dos valores SHAP  = {shap_values.shape}")

valor-base E[P(Maligno)] = 0.340
formato dos valores SHAP  = (40, 30)


## Explicando Um Paciente: O Waterfall Plot

O **Waterfall** decompõe uma única previsão, mostrando como cada *feature* move o resultado do valor-base até a previsão final. Vamos explicar o **primeiro paciente maligno** do nosso lote de treino: um caso **maligno**, corretamente classificado, com **98,7%** de probabilidade — inequívoco, ótimo para ver a mecânica do gráfico. (Usamos o treino: o cofre segue lacrado.)

#### Código 19.2: Waterfall de Uma Previsão


In [ ]:
paciente = 0
explicacao = shap.Explanation(values=shap_values[paciente],
                              base_values=explainer.expected_value,
                              data=amostra.iloc[paciente].values,
                              feature_names=list(FEATURES_MORFOLOGICAS))
shap.plots.waterfall(explicacao, max_display=10)

![Waterfall — paciente maligno do treino](media/figura_19_1.png)

Partindo do valor-base **0,340**, a previsão sobe até **0,987**. Quem empurra para **maligno**: `worst concave points` (+0,183, o maior impacto), `worst concavity` (+0,154) e `mean concavity` (+0,059) — tudo ligado à **irregularidade do contorno**. Neste caso, praticamente **nenhuma** *feature* puxa para benigno — é um tumor inequivocamente maligno. A soma de todas as contribuições mais o valor-base fecha exatamente em 0,987: a conta bate.

## A Visão Agregada: Beeswarm e Bar

#### Código 19.3: Beeswarm e Importância Média


In [ ]:
# Beeswarm: cada ponto e um paciente; cor = valor da feature; eixo x = impacto.
shap.summary_plot(shap_values, amostra.values,
                  feature_names=list(FEATURES_MORFOLOGICAS), max_display=12)

# Bar: importancia media (|SHAP| medio) por feature.
shap.summary_plot(shap_values, amostra.values, plot_type="bar",
                  feature_names=list(FEATURES_MORFOLOGICAS), max_display=12)

![Beeswarm](media/figura_19_2.png)

![Importância média](media/figura_19_3.png)

O **beeswarm** revela padrões globais a partir das explicações locais: para `worst concave points`, valores altos (pontos vermelhos) empurram para maligno, valores baixos (azuis) para benigno — a lógica do modelo confirmada. O **bar** ordena a importância média: `worst concave points` (0,053), `worst concavity` (0,035) e `worst radius` (0,030) no topo, coerente com a Permutation Importance do capítulo anterior — duas técnicas independentes contando a mesma história.

> **📘 PARA SABER — Cuidado com "global" a partir de poucas amostras**
>
> Nosso beeswarm agrega 40 pacientes, por causa do custo do `KernelExplainer`. Isso ilustra bem os padrões, mas conclusões verdadeiramente globais pedem mais amostras. Para modelos de árvore existe o `TreeExplainer`, ordens de magnitude mais rápido, que permite explicar o dataset inteiro.

> **📌 CHECKLIST DO CAPÍTULO 19**
>
> ✅ Entende a intuição do SHAP (teoria dos jogos) e a **eficiência local** (a conta fecha).
>
> ✅ Sabe interpretar um **Waterfall** (uma previsão), um **Beeswarm** (visão agregada) e um **Bar** (importância média).
>
> ✅ Executou os códigos e viu a explicação de um paciente somar exatamente à sua probabilidade prevista.
>
> ✅ Notou a concordância entre SHAP e Permutation Importance.
>
> **⚠️ CRÍTICO:** O SHAP é o padrão-ouro da interpretabilidade: base teórica sólida e visualizações intuitivas para construir confiança e depurar modelos.

Os gráficos do SHAP eram hipnotizantes. Eu podia ver a "mente" do modelo em ação: para cada paciente, uma narrativa de como suas características se combinaram num diagnóstico. Podia sentar ao lado do chefe da oncologia, mostrar o Waterfall de um paciente e dizer: "o modelo aponta maligno, principalmente pela alta concavidade e pelos pontos côncavos do núcleo; a compacidade puxa um pouco para o outro lado, mas não o suficiente".

O SHAP era rigoroso. Mas havia outra filosofia de explicação local — mais simples, que eu poderia esboçar num guardanapo. Uma abordagem chamada LIME. Seria minha última parada na interpretabilidade.
